In [ ]:
import os, sys, subprocess, importlib.metadata as metadata

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

PINS = {
    "transformers": "4.57.3",
    "trl": "0.24.0",
    "peft": "0.17.1",
    "accelerate": "1.11.0",
    "datasets": "4.3.0",
    "evalplus": "0.3.1",
}

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "No Kaggle GPU detected. Open Notebook Settings and select a GPU accelerator."
    )
print("GPU:", torch.cuda.get_device_name(0))

packages = [f"{name}=={version}" for name, version in PINS.items() if name != "evalplus"]
packages += ["pandas>=2.2,<3", "matplotlib>=3.9,<4", "psutil"]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *packages],
    check=True
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "evalplus==0.3.1", "--no-deps"],
    check=True
)

stale = []
for name, version in PINS.items():
    try:
        installed = metadata.version(name)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        stale.append(f"{name}: expected {version}, found {installed}")
if stale:
    print("Package version differences:", stale)

from transformers import AutoTokenizer, AutoModelForCausalLM, TrainerCallback, set_seed
from peft import LoraConfig, PeftModel, get_peft_model
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset
import pandas as pd
import matplotlib.pyplot as plt
import psutil

DEVICE = "cuda"
USE_BF16 = torch.cuda.is_bf16_supported()
USE_FP16 = not USE_BF16
MODEL_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
DEVICE_NAME = torch.cuda.get_device_name(0)

print("Kaggle GPU setup ready")
print("Device:", DEVICE_NAME)
print("Base dtype:", MODEL_DTYPE)
print("BF16:", USE_BF16, "| FP16:", USE_FP16)
print("Transformers:", metadata.version("transformers"))
print("TRL:", metadata.version("trl"))
print("PEFT:", metadata.version("peft"))
print("Datasets:", metadata.version("datasets"))
print("EvalPlus:", metadata.version("evalplus"))


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pathos 0.3.5 requires dill>=0.4.1, but you have dill 0.4.0 which is incompatible.
pathos 0.3.5 requires multiprocess>=0.70.19, but you have multiprocess 0.70.16 which is incompatible.
s3fs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.9.0 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.9.0 which is incompatible.


Kaggle GPU setup ready
Device: Tesla T4
Base dtype: torch.bfloat16
BF16: True | FP16: False
Transformers: 4.57.3
TRL: 0.24.0
PEFT: 0.17.1
Datasets: 4.3.0
EvalPlus: 0.3.1


In [2]:
import torch
import google.protobuf
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainerCallback, set_seed
from peft import LoraConfig, PeftModel, get_peft_model
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset
import pandas as pd
import matplotlib.pyplot as plt
import psutil

assert torch.cuda.is_available(), "Enable a GPU accelerator in Kaggle."
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Protobuf:", google.protobuf.__version__)
print("All required imports successful.")


PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4
Protobuf: 5.29.5
All required imports successful.


In [ ]:
import ast, copy, csv, gzip, hashlib, html, itertools, json, math, os, random, re
import statistics, tempfile, textwrap, time, urllib.request, gc, subprocess
from pathlib import Path
from functools import lru_cache
from datetime import datetime, timezone
from IPython.display import display
import importlib.metadata as metadata
import torch
import psutil

if 'DEVICE' not in globals():
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if 'USE_BF16' not in globals():
    USE_BF16 = DEVICE == 'cuda' and torch.cuda.is_bf16_supported()
if 'USE_FP16' not in globals():
    USE_FP16 = DEVICE == 'cuda' and not USE_BF16
if 'MODEL_DTYPE' not in globals():
    MODEL_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16 if USE_FP16 else torch.float32
if 'DEVICE_NAME' not in globals():
    DEVICE_NAME = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'

MODEL_NAME = 'Qwen/Qwen2.5-Coder-0.5B-Instruct'
TASK_IDS = [f'HumanEval/{index}' for index in range(24)]
SEEDS = [17, 42, 73]
STEPS_PER_STAGE = 5
STAGES = 2
WEAK_TEST_COUNT = 1
CHALLENGE_TEST_COUNT = 8
CHALLENGE_CATEGORIES = ('boundary','local','scale','structure')
EVAL_SAMPLES_PER_TASK = 2
PROBE_SAMPLES_PER_TASK = 2
MAX_NEW_TOKENS = 256
NUM_GENERATIONS = 4
WEIGHTS = {'Pw':.40, 'Pc':.35, 'Mw':.15, 'V':.10}
MODEL_REVISION = None
run_name = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')

def create_output_directory():
    path = Path('/kaggle/working') / 'BAGR_project_results' / run_name
    if not str(Path('/kaggle/working')).startswith('/kaggle'):
        raise RuntimeError('Unexpected Kaggle workspace path.')
    path.mkdir(parents=True,exist_ok=False)
    for folder in ['data','candidates','checkpoints','logs','results','figures']:
        (path / folder).mkdir(exist_ok=True)
    return path

RESUME_RUN_DIR = None

if RESUME_RUN_DIR is None:
    OUTPUT_DIR = create_output_directory()
else:
    OUTPUT_DIR = Path(RESUME_RUN_DIR).resolve()
    kaggle_working = Path('/kaggle/working').resolve()
    if kaggle_working not in OUTPUT_DIR.parents:
        raise ValueError('RESUME_RUN_DIR must be inside /kaggle/working.')
    required_dirs = ['candidates', 'checkpoints', 'results', 'logs']
    missing_dirs = [name for name in required_dirs if not (OUTPUT_DIR / name).exists()]
    if missing_dirs:
        raise FileNotFoundError(f'Restored run is missing folders: {missing_dirs}')
    print('Resuming evaluation from saved run:', OUTPUT_DIR)

def clear_device_cache():
    gc.collect()
    if DEVICE=='cuda':
        torch.cuda.empty_cache()
    elif DEVICE=='mps':
        torch.mps.empty_cache()

def reset_memory_stats():
    if DEVICE=='cuda':
        torch.cuda.reset_peak_memory_stats()

def memory_stats():
    return {'Peak CUDA GB':torch.cuda.max_memory_allocated()/1024**3 if DEVICE=='cuda' else None,
            'MPS allocated GB':torch.mps.current_allocated_memory()/1024**3 if DEVICE=='mps' else None,
            'Process RSS GB':psutil.Process().memory_info().rss/1024**3}

def worker_environment():
    keys = ['PATH','SYSTEMROOT','WINDIR','TEMP','TMP','TMPDIR','HOME','USERPROFILE',
            'APPDATA','LOCALAPPDATA','HOMDRIVE','HOMEPATH']
    env = {key:os.environ[key] for key in keys if key in os.environ}
    env.update(LANG='C.UTF-8',OMP_NUM_THREADS='1')
    return env

def save_json(filename, value):
    path = OUTPUT_DIR / filename
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,ensure_ascii=False,allow_nan=False),encoding='utf-8')

def save_csv(filename, rows):
    if rows:
        path = OUTPUT_DIR / filename
        path.parent.mkdir(parents=True,exist_ok=True)
        pd.DataFrame(rows).to_csv(path,index=False)

def display_table(rows, title=''):
    print(title)
    display(pd.DataFrame(rows))


def save_checkpoint_manifest():
    root = OUTPUT_DIR / 'checkpoints'
    records = []
    if root.exists():
        for path in sorted(root.rglob('*')):
            if path.is_file():
                digest = hashlib.sha256()
                with path.open('rb') as handle:
                    for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                        digest.update(chunk)
                records.append({
                    'relative_path': str(path.relative_to(OUTPUT_DIR)),
                    'bytes': path.stat().st_size,
                    'sha256': digest.hexdigest()
                })
    save_json('checkpoint_manifest.json', {
        'workspace': str(OUTPUT_DIR.resolve()),
        'checkpoint_file_count': len(records),
        'files': records
    })
    return records

pinned_packages = ['transformers', 'trl', 'peft', 'accelerate', 'datasets', 'evalplus']
settings = {'model':MODEL_NAME,'tasks':TASK_IDS,'seeds':SEEDS,
    'steps_per_stage':STEPS_PER_STAGE,'stages':STAGES,
    'trained_steps_per_variant':STEPS_PER_STAGE*STAGES,
    'weak_tests':WEAK_TEST_COUNT,'challenge_tests':CHALLENGE_TEST_COUNT,
    'challenge_categories':CHALLENGE_CATEGORIES,
    'eval_samples_per_task':EVAL_SAMPLES_PER_TASK,
    'probe_samples_per_task':PROBE_SAMPLES_PER_TASK,
    'generation_max_tokens':MAX_NEW_TOKENS,'num_generations':NUM_GENERATIONS,
    'temperature':.8,'top_p':.95,'top_k':0,'repetition_penalty':1.0,
    'weights':WEIGHTS,'packages':{name:metadata.version(name) for name in pinned_packages},
    'torch':torch.__version__,'device':DEVICE,'device_name':DEVICE_NAME,
    'base_dtype':str(MODEL_DTYPE),'bf16':USE_BF16,'fp16':USE_FP16}
save_json('settings.json',settings)
print('Results directory:',OUTPUT_DIR.resolve())
print('Tasks:',len(TASK_IDS),'Seeds:',SEEDS,'Precision:',MODEL_DTYPE)

Results directory: /kaggle/working/BAGR_project_results/20261102_125023_584374
Tasks: 24 Seeds: [17, 42, 73] Precision: torch.bfloat16


### 3 — Load the public HumanEval training tasks

Only the original public HumanEval prompts, canonical solutions, and public tests are used during training. HumanEval+ remains unopened until the independent final evaluator runs.


In [4]:
PUBLIC_URL = 'https://raw.githubusercontent.com/openai/human-eval/master/data/HumanEval.jsonl.gz'
with urllib.request.urlopen(PUBLIC_URL, timeout=60) as response:
    public_bytes = response.read()
public_tasks = {row['task_id']:row for row in
                (json.loads(line) for line in gzip.decompress(public_bytes).decode().splitlines())}
problems = {task:public_tasks[task] for task in TASK_IDS}
assert all('plus_input' not in problem for problem in problems.values())
save_json('data/public_tasks.json', problems)
save_json('data/provenance.json', {'source':PUBLIC_URL, 'sha256':hashlib.sha256(public_bytes).hexdigest()})
display_table([{'Task ID':task,'Function':problems[task]['entry_point']} for task in TASK_IDS],f'{len(TASK_IDS)} official public tasks')


24 official public tasks


,Task ID,Function
0,HumanEval/0,has_close_elements
1,HumanEval/1,separate_paren_groups
2,HumanEval/2,truncate_number
3,HumanEval/3,below_zero
4,HumanEval/4,mean_absolute_deviation
5,HumanEval/5,intersperse
6,HumanEval/6,parse_nested_parens
7,HumanEval/7,filter_by_substring
8,HumanEval/8,sum_product
9,HumanEval/9,rolling_max


In [5]:
def normalize_json(value):
    return json.loads(json.dumps(value,ensure_ascii=False,allow_nan=False))

def collect_public_tests(problem):
    namespace = {}
    exec(problem['prompt'] + problem['canonical_solution'], namespace)
    reference = namespace[problem['entry_point']]
    inputs, expected = [], []

    def recording_reference(*arguments):
        inputs.append(normalize_json(list(arguments)))
        result=reference(*copy.deepcopy(arguments))
        expected.append(normalize_json(result))
        return result

    exec(problem['test'],namespace)
    namespace['check'](recording_reference)
    return inputs, expected

public_inputs, public_expected = {}, {}
for task in TASK_IDS:
    public_inputs[task], public_expected[task] = collect_public_tests(problems[task])
assert all(len(public_inputs[task])>=WEAK_TEST_COUNT for task in TASK_IDS)
display_table([{'Task ID':task,'Public tests':len(public_inputs[task]),
                'Weak-5 eligible':len(public_inputs[task])>=5} for task in TASK_IDS],
              'Available public tests')


Available public tests


,Task ID,Public tests,Weak-5 eligible
0,HumanEval/0,7,True
1,HumanEval/1,4,False
2,HumanEval/2,3,False
3,HumanEval/3,6,True
4,HumanEval/4,3,False
5,HumanEval/5,3,False
6,HumanEval/6,3,False
7,HumanEval/7,4,False
8,HumanEval/8,5,True
9,HumanEval/9,4,False


### 4 — Bounded candidate executor

Generated code runs in isolated subprocesses with time and memory limits. Unlike the earlier four-task version, any finite JSON-serializable result type is accepted, so all selected HumanEval tasks can be scored.


In [6]:
WORKER_CODE = r'''
import ast, builtins, copy, json, math, signal, sys, time
try:
    import resource
except ImportError:
    resource = None
HAS_TIMERS = hasattr(signal,'SIGALRM') and hasattr(signal,'setitimer')
ALLOWED_IMPORTS = {'math','typing','collections','itertools','functools','string','statistics','heapq','bisect'}
FORBIDDEN = {'open','exec','eval','compile','globals','locals','vars','getattr','setattr','delattr','input',
             'os','sys','subprocess','socket','builtins','resource','signal','help','exit','quit','__import__'}

def import_allowed(name,globals=None,locals=None,fromlist=(),level=0):
    if level or name not in ALLOWED_IMPORTS:
        raise ImportError('Import not allowed')
    return builtins.__import__(name,globals,locals,fromlist,level)

def timeout(signum,frame):
    raise TimeoutError('Execution timeout')

def arm(seconds):
    if HAS_TIMERS:
        signal.setitimer(signal.ITIMER_REAL,seconds)

def main():
    request = json.load(sys.stdin)
    if resource:
        cpu = max(2,math.ceil(len(request['inputs'])*.5)+2)
        resource.setrlimit(resource.RLIMIT_CPU,(cpu,cpu))
        resource.setrlimit(resource.RLIMIT_FSIZE,(1000000,1000000))
        if sys.platform.startswith('linux'):
            resource.setrlimit(resource.RLIMIT_AS,(256*1024**2,256*1024**2))
    names = ('abs all any bool chr dict divmod enumerate filter float int isinstance iter len list map max min next '
             'ord pow range repr reversed round set sorted str sum tuple type zip Exception ValueError TypeError '
             'ZeroDivisionError StopIteration AssertionError').split()
    allowed = {name:getattr(builtins,name) for name in names}
    allowed.update(__import__=import_allowed,print=lambda *args,**kwargs:None)
    namespace = {'__builtins__':allowed,'__name__':'candidate'}
    if HAS_TIMERS:
        signal.signal(signal.SIGALRM,timeout)
    results = []
    try:
        tree = ast.parse(request['source'])
        for node in ast.walk(tree):
            if isinstance(node,ast.Name) and (node.id in FORBIDDEN or node.id.startswith('__')):
                raise ValueError('Disallowed name')
            if isinstance(node,ast.Attribute) and (node.attr.startswith('_') or node.attr in FORBIDDEN):
                raise ValueError('Disallowed attribute')
            if isinstance(node,ast.Import) and any(a.name not in ALLOWED_IMPORTS for a in node.names):
                raise ValueError('Disallowed import')
            if isinstance(node,ast.ImportFrom) and (node.level or node.module not in ALLOWED_IMPORTS):
                raise ValueError('Disallowed import')
        started = time.perf_counter()
        arm(.5)
        exec(compile(tree,'<candidate>','exec'),namespace)
        function = namespace[request['entry_point']]
        if time.perf_counter()-started > .5:
            raise TimeoutError('Execution timeout')
    except BaseException as error:
        results = [{'status':'timeout' if isinstance(error,TimeoutError) else 'invalid_code',
                    'output':None,'error':type(error).__name__} for _ in request['inputs']]
    else:
        arm(0)
        for arguments in request['inputs']:
            try:
                started = time.perf_counter()
                arm(.5)
                output = function(*copy.deepcopy(arguments))
                encoded = json.dumps(output,allow_nan=False)
                if time.perf_counter()-started > .5:
                    raise TimeoutError('Execution timeout')
                if len(encoded)>100000:
                    raise ValueError('Output too large')
                results.append({'status':'ok','output':json.loads(encoded),'error':None})
            except BaseException as error:
                results.append({'status':'timeout' if isinstance(error,TimeoutError) else 'runtime_error',
                                'output':None,'error':type(error).__name__})
            finally:
                arm(0)
    arm(0)
    json.dump(results,sys.stdout)

if __name__=='__main__':
    main()

'''

def clean_code(completion):
    text = str(completion).strip('\r\n')
    blocks = re.findall(r'```(?:python)?\s*\n(.*?)```', text, flags=re.DOTALL)
    return blocks[0].strip('\r\n') if blocks else text

def complete_source(task, completion):
    code_text = clean_code(completion)
    problem = problems[task]
    try:
        tree = ast.parse(textwrap.dedent(code_text))
        if any(isinstance(node, ast.FunctionDef) and node.name == problem['entry_point'] for node in tree.body):
            imports = problem['prompt'].split('def ' + problem['entry_point'])[0]
            return imports + textwrap.dedent(code_text)
    except SyntaxError:
        pass
    body = code_text if code_text.startswith((' ', '\t')) else textwrap.indent(code_text, '    ')
    return problem['prompt'] + body + '\n'


def run_worker_request(request):
    try:
        with tempfile.TemporaryDirectory() as directory:
            worker = Path(directory)/'worker.py'
            worker.write_text(WORKER_CODE,encoding='utf-8')
            process = subprocess.run([sys.executable,'-X','utf8','-I','-S',str(worker)],
                input=json.dumps(request),capture_output=True,text=True,encoding='utf-8',cwd=directory,
                env=worker_environment(),timeout=3+.5*len(request['inputs']))
        if process.returncode!=0:
            raise RuntimeError(process.stderr[-200:])
        results = json.loads(process.stdout)
        if len(results)!=len(request['inputs']):
            raise RuntimeError('Incomplete execution results')
        return results
    except subprocess.TimeoutExpired:
        return [{'status':'timeout','output':None,'error':'Wall timeout'} for _ in request['inputs']]
    except Exception as error:
        return [{'status':'worker_error','output':None,'error':str(error)[:200]} for _ in request['inputs']]

@lru_cache(maxsize=512)
def cached_execution(task,completion,inputs_json):
    inputs = json.loads(inputs_json)
    request = {'source':complete_source(task,completion),'entry_point':problems[task]['entry_point'],'inputs':inputs}
    if os.name=='nt':
        # Windows has no SIGALRM: isolate each input so a hanging call cannot hide other scores.
        return [result for arguments in inputs
                for result in run_worker_request(dict(request,inputs=[arguments]))]
    return run_worker_request(request)

def execute_candidate(task,completion,inputs):
    return copy.deepcopy(cached_execution(task,str(completion),json.dumps(inputs,sort_keys=True)))

def trusted_outputs(task,inputs):
    results = execute_candidate(task,problems[task]['canonical_solution'],inputs)
    assert all(result['status']=='ok' for result in results),(task,results)
    return [result['output'] for result in results]

print('Bounded execution helper ready for',os.name+'.')


Bounded execution helper ready for posix.


In [ ]:
import sys
if "'sys'" not in WORKER_CODE.split("allowed = {")[1].split("}")[0]:
    globals()['WORKER_CODE'] = WORKER_CODE.replace(
        "allowed = {name:getattr(builtins,name) for name in names}",
        "allowed = {name:getattr(builtins,name) for name in names}\n    allowed['sys'] = sys"
    )
    # Clear the lru_cache of cached_execution to apply the patch
    cached_execution.cache_clear()

for task in TASK_IDS:
    observed = trusted_outputs(task,public_inputs[task])
    assert observed==public_expected[task],(task,observed,public_expected[task])
print('All',len(TASK_IDS),'public reference suites validated by the bounded executor.')

All 24 public reference suites validated by the bounded executor.


### 5 — Load Qwen and create matched baseline cohorts

Each seed gets its own E1 samples and policy probes. All later methods use the same prompts and generation settings, and the resolved model revision is pinned for every reload.


In [ ]:


import torch
import json
import gc
import os
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from datasets import Dataset


if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is unavailable. Enable GPU in notebook settings."
    )

DEVICE = "cuda"
DEVICE_NAME = torch.cuda.get_device_name(0)

print("GPU:", DEVICE_NAME)
print("CUDA:", torch.version.cuda)
print("PyTorch:", torch.__version__)


MODEL_NAME = "Qwen/Qwen2.5-Coder-0.5B-Instruct"

if MODEL_NAME is None:
    raise ValueError("MODEL_NAME is missing.")

MODEL_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

print("Model dtype:", MODEL_DTYPE)


print("\nLoading Qwen base model...")

model_load_options = {
    "dtype": MODEL_DTYPE,
    "low_cpu_mem_usage": True
}

if MODEL_REVISION is not None:
    model_load_options["revision"] = MODEL_REVISION

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    **model_load_options
)

model = model.to(DEVICE)

MODEL_REVISION = getattr(model.config, "_commit_hash", None)

if not MODEL_REVISION:
    raise RuntimeError(
        "Could not resolve the model commit hash."
    )

model.config.use_cache = False

print("Model loaded successfully.")
print("Model revision:", MODEL_REVISION)
print("Model device:", next(model.parameters()).device)


print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION
)

tokenizer.padding_side = "left"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.pad_token_id

if model.generation_config is not None:
    model.generation_config.pad_token_id = tokenizer.pad_token_id

print("Tokenizer ready.")
print("PAD token:", tokenizer.pad_token)
print("PAD token ID:", tokenizer.pad_token_id)

# Save exact model revision
save_json(
    "model_revision.json",
    {
        "model": MODEL_NAME,
        "revision": MODEL_REVISION,
        "dtype": str(MODEL_DTYPE),
        "device": DEVICE,
        "device_name": DEVICE_NAME
    }
)


messages = {
    task: [
        {
            "role": "user",
            "content":
                "Solve this Python problem. Return only the complete "
                "Python function and necessary imports. "
                "Use the exact function name and specified behavior.\n\n"
                + problems[task]["prompt"]
        }
    ]
    for task in TASK_IDS
}

train_dataset = Dataset.from_list([
    {
        "task_id": task,
        "prompt": messages[task]
    }
    for task in TASK_IDS
])

print("\nTraining dataset created:", len(train_dataset))

def load_base_model():

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        revision=MODEL_REVISION,
        dtype=MODEL_DTYPE,
        low_cpu_mem_usage=True
    )

    model = model.to(DEVICE)

    model.config.use_cache = False
    model.config.pad_token_id = tokenizer.pad_token_id

    if model.generation_config is not None:
        model.generation_config.pad_token_id = tokenizer.pad_token_id

    return model


def sample_policy(model, count, seed, method):

    records = []

    was_training = model.training
    model.eval()

    try:

        # Reproducible CPU and GPU random states
        devices = [torch.cuda.current_device()]

        with torch.random.fork_rng(devices=devices):

            set_seed(seed)

            for task in TASK_IDS:

                prompt = tokenizer.apply_chat_template(
                    messages[task],
                    tokenize=False,
                    add_generation_prompt=True
                )

                inputs = tokenizer(
                    prompt,
                    return_tensors="pt",
                    add_special_tokens=False
                ).to(DEVICE)

                for index in range(count):

                    with torch.inference_mode():

                        generated = model.generate(
                            **inputs,
                            do_sample=True,
                            temperature=0.8,
                            top_p=0.95,
                            top_k=0,
                            repetition_penalty=1.0,
                            max_new_tokens=MAX_NEW_TOKENS,
                            use_cache=True,
                            pad_token_id=tokenizer.pad_token_id,
                            eos_token_id=tokenizer.eos_token_id
                        )

                    new_tokens = generated[
                        0,
                        inputs["input_ids"].shape[-1]:
                    ]

                    completion = tokenizer.decode(
                        new_tokens,
                        skip_special_tokens=True
                    )

                    records.append({
                        "method": method,
                        "seed": seed,
                        "task_id": task,
                        "sample_id": index,
                        "completion": completion,
                        "token_limit_reached":
                            len(new_tokens) >= MAX_NEW_TOKENS
                    })

    finally:
        model.train(was_training)

    return records



def save_candidates(filename, candidates):

    path = OUTPUT_DIR / filename
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as file:

        for candidate in candidates:

            record = dict(
                candidate,
                solution=complete_source(
                    candidate["task_id"],
                    candidate["completion"]
                )
            )

            file.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )



print("\n" + "=" * 60)
print("E1: BASELINE MODEL GENERATION")
print("=" * 60)

baseline_candidates_by_seed = {}
baseline_probes_by_seed = {}

model.eval()

for seed in SEEDS:

    print(f"\nGenerating baseline candidates for seed {seed}")

    candidates = sample_policy(
        model,
        EVAL_SAMPLES_PER_TASK,
        seed + 1000,
        "E1"
    )

    probes = sample_policy(
        model,
        PROBE_SAMPLES_PER_TASK,
        seed + 2000,
        "E1_probe"
    )

    baseline_candidates_by_seed[seed] = candidates
    baseline_probes_by_seed[seed] = probes

    save_candidates(
        f"candidates/seed_{seed}/E1.jsonl",
        candidates
    )

    save_candidates(
        f"candidates/seed_{seed}/baseline_probes.jsonl",
        probes
    )

    print(
        f"Seed {seed}: "
        f"{len(candidates)} evaluation programs | "
        f"{len(probes)} probe programs"
    )


del model

gc.collect()
torch.cuda.empty_cache()


total_candidates = sum(
    len(values)
    for values in baseline_candidates_by_seed.values()
)

total_probes = sum(
    len(values)
    for values in baseline_probes_by_seed.values()
)

expected = len(SEEDS) * len(TASK_IDS) * EVAL_SAMPLES_PER_TASK

assert total_candidates == expected

print("\n" + "=" * 60)
print("E1 BASELINE GENERATION COMPLETED")
print("=" * 60)

print("GPU:", DEVICE_NAME)
print("Total tasks:", len(TASK_IDS))
print("Total seeds:", len(SEEDS))
print("Baseline candidates:", total_candidates)
print("Baseline probes:", total_probes)
print("Model revision:", MODEL_REVISION)
print("Results directory:", OUTPUT_DIR.resolve())
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3 , 3), "GB")

print("\nE1 successfully completed.")


GPU: Tesla T4
CUDA: 12.8
PyTorch: 2.11.0+cu128
Model dtype: torch.bfloat16

Loading Qwen base model...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded successfully.
Model revision: ea3f2471cf1b1f0db85067f1ef93848e38e88c25
Model device: cuda:0

Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer ready.
PAD token: <|endoftext|>
PAD token ID: 151643

Training dataset created: 24

E1: BASELINE MODEL GENERATION

Generating baseline candidates for seed 17
Seed 17: 48 evaluation programs | 48 probe programs

Generating baseline candidates for seed 42
Seed 42: 48 evaluation programs | 48 probe programs

Generating baseline candidates for seed 73
Seed 73: 48 evaluation programs | 48 probe programs

E1 BASELINE GENERATION COMPLETED
GPU: Tesla T4
Total tasks: 24
Total seeds: 3
Baseline candidates: 144
Baseline probes: 144
Model revision: ea3f2471cf1b1f0db85067f1ef93848e38e88c25
Results directory: /kaggle/working/BAGR_project_results/20261102_125023_584374
GPU memory allocated: 0.009 GB

E1 successfully completed.


### 6 — Weak verifier and balanced challenge generation

Challenge inputs are derived from valid public inputs by four mutation families: boundary, local perturbation, scale, and structure. Every retained input is distinct from the public suite and must execute successfully under the canonical solution. Selection is round-robin by category, which prevents one easy mutation family from dominating a task.


In [ ]:
import copy
import json
import random


print("=" * 65)
print("BAGR CHALLENGE TEST GENERATION")
print("=" * 65)


def correctness(task, completion, inputs, expected):

    if not inputs:
        return 0.0, []

    results = execute_candidate(task, completion, inputs)

    passed = sum(
        result["status"] == "ok" and result["output"] == gold
        for result, gold in zip(results, expected)
    )

    return passed / len(inputs), results


def weak_correctness(task, completion, count=WEAK_TEST_COUNT):

    total = (
        len(public_inputs[task])
        if count == "all"
        else count
    )

    if total > len(public_inputs[task]):
        return None

    return correctness(
        task,
        completion,
        public_inputs[task][:total],
        public_expected[task][:total]
    )[0]



def value_variants(value, category, rng):

    if isinstance(value, bool):

        choices = {
            "boundary": [False, True],
            "local": [not value],
            "scale": [int(value) + 1],
            "structure": [str(value).lower()]
        }

    elif isinstance(value, int):

        choices = {
            "boundary": [0, 1, -1, 999, -999],
            "local": [value - 1, value + 1, value + 2],
            "scale": [value * 2, value * 3, max(1, value // 2)],
            "structure": [-value, abs(value) + 7, value + 100]
        }

    elif isinstance(value, float):

        choices = {
            "boundary": [0.0, 1.0, -1.0, 999.0, 0.0001],
            "local": [value - 0.5, value + 0.5, value + 1.0],
            "scale": [value * 2.0, value / 2.0],
            "structure": [-value, abs(value) + 0.25, value + 100.0]
        }

    elif isinstance(value, str):

        tokens = value.split()

        first = tokens[0] if tokens else ""
        last = tokens[-1] if tokens else ""

        choices = {
            "boundary": ["", " ", first, last, "A" * 50],
            "local": [
                value[:-1],
                value[1:],
                " ".join(tokens[:-1]),
                " ".join(tokens + [first])
            ],
            "scale": [
                value + value,
                value * 3,
                " ".join(tokens + tokens)
            ],
            "structure": [
                value[::-1],
                value.swapcase(),
                " ".join(reversed(tokens))
            ]
        }

    elif isinstance(value, list):

        first = copy.deepcopy(value[0]) if value else 0

        choices = {
            "boundary": [[], [first], [first, first]],
            "local": [value[:-1], value + [first]],
            "scale": [
                value + copy.deepcopy(value),
                value * 3
            ],
            "structure": [
                list(reversed(value)),
                sorted(value, key=repr)
            ]
        }

    elif isinstance(value, dict):

        items = list(value.items())

        choices = {
            "boundary": [{}],
            "local": [dict(items[:-1])],
            "scale": [
                dict(items + [(str(len(items)), len(items))])
            ],
            "structure": [
                dict(reversed(items))
            ]
        }

    elif value is None:

        choices = {
            "boundary": [0],
            "local": [""],
            "scale": [False],
            "structure": [[]]
        }

    else:

        choices = {
            name: []
            for name in CHALLENGE_CATEGORIES
        }

    unique = []

    for candidate in choices[category]:

        try:

            candidate = normalize_json(candidate)

            if candidate != value and candidate not in unique:
                unique.append(candidate)

        except (TypeError, ValueError):
            pass

    rng.shuffle(unique)

    return unique



def generate_challenge_pool(task, seed):

    rng = random.Random(seed)

    public_keys = {
        json.dumps(args, sort_keys=True, ensure_ascii=False)
        for args in public_inputs[task]
    }

    proposals = []
    seen = set(public_keys)

    bases = copy.deepcopy(public_inputs[task])

    for cycle in range(6):

        for base in bases:

            for category in CHALLENGE_CATEGORIES:

                for position, value in enumerate(base):

                    replacements = value_variants(
                        value,
                        category,
                        rng
                    )

                    for replacement in replacements:

                        arguments = copy.deepcopy(base)

                        arguments[position] = replacement

                        key = json.dumps(
                            arguments,
                            sort_keys=True,
                            ensure_ascii=False
                        )

                        if key not in seen:

                            seen.add(key)

                            proposals.append(
                                (arguments, category)
                            )

            if len(base) > 1:

                structural_candidates = [
                    (
                        "structure",
                        list(reversed(copy.deepcopy(base)))
                    ),
                    (
                        "scale",
                        copy.deepcopy(base) + copy.deepcopy(base)
                    )
                ]

                for category, arguments in structural_candidates:

                    key = json.dumps(
                        arguments,
                        sort_keys=True,
                        ensure_ascii=False
                    )

                    if key not in seen:

                        seen.add(key)

                        proposals.append(
                            (arguments, category)
                        )

        rng.shuffle(bases)

    inputs = [
        arguments
        for arguments, _ in proposals
    ]

    reference_code = problems[task]["canonical_solution"]

    results = execute_candidate(
        task,
        reference_code,
        inputs
    )

    pool = []

    for (arguments, category), result in zip(proposals, results):

        if result["status"] == "ok":

            pool.append({
                "task_id": task,
                "input": arguments,
                "expected_output": result["output"],
                "category": category,
                "method": "seeded public-input mutation",
                "reference_validated": True,
                "official_humaneval_plus_test": False
            })

    unique_keys = {
        json.dumps(
            record["input"],
            sort_keys=True,
            ensure_ascii=False
        )
        for record in pool
    }

    assert len(unique_keys) == len(pool), (
        f"Duplicate challenge inputs for {task}"
    )

    return pool



def choose_challenges(pool, policy_candidates=None):

    selected = {}

    for task in TASK_IDS:

        ranked = copy.deepcopy(pool[task])

        for record in ranked:

            record["candidate_failures"] = 0
            record["selection_score"] = 0

        for candidate in policy_candidates or []:

            if candidate["task_id"] != task:
                continue

            outcomes = execute_candidate(
                task,
                candidate["completion"],
                [
                    record["input"]
                    for record in ranked
                ]
            )

            if all(
                outcome["status"] in {
                    "invalid_code",
                    "worker_error"
                }
                for outcome in outcomes
            ):
                continue

            weak_pass = (
                weak_correctness(
                    task,
                    candidate["completion"]
                ) == 1
            )

            for record, outcome in zip(ranked, outcomes):

                if (
                    outcome["status"] != "ok"
                    or outcome["output"] != record["expected_output"]
                ):

                    record["candidate_failures"] += 1

                    record["selection_score"] += (
                        2 if weak_pass else 1
                    )

        buckets = {
            category: []
            for category in CHALLENGE_CATEGORIES
        }

        for record in ranked:

            buckets[record["category"]].append(record)

        for records in buckets.values():

            records.sort(
                key=lambda record: (
                    -record["selection_score"],
                    json.dumps(
                        record["input"],
                        sort_keys=True
                    )
                )
            )

        chosen = []

        while len(chosen) < CHALLENGE_TEST_COUNT:

            progressed = False

            for category in CHALLENGE_CATEGORIES:

                if (
                    buckets[category]
                    and len(chosen) < CHALLENGE_TEST_COUNT
                ):

                    chosen.append(
                        buckets[category].pop(0)
                    )

                    progressed = True

            if not progressed:
                break

        if len(chosen) < CHALLENGE_TEST_COUNT:

            remaining = {
                name: len(records)
                for name, records in buckets.items()
            }

            raise ValueError(
                f"{task}: Only {len(chosen)} distinct validated "
                f"challenge tests available. "
                f"Required: {CHALLENGE_TEST_COUNT}. "
                f"Remaining: {remaining}. "
                "Improve generation or revise the challenge budget."
            )

        selected[task] = chosen

    return selected


challenge_pools = {}

static_challenges_by_seed = {}

policy_challenges_by_seed = {}

challenge_validation = []

for seed in SEEDS:

    print(f"\nGenerating challenge tests for seed={seed}")

    pool = {
        task: generate_challenge_pool(
            task,
            seed * 1000 + index
        )
        for index, task in enumerate(TASK_IDS)
    }

    challenge_pools[seed] = pool

    static_challenges_by_seed[seed] = choose_challenges(
        pool
    )

    policy_challenges_by_seed[seed] = choose_challenges(
        pool,
        baseline_probes_by_seed[seed]
    )

    save_json(
        f"data/seed_{seed}/static_challenges.json",
        static_challenges_by_seed[seed]
    )

    save_json(
        f"data/seed_{seed}/policy_challenges.json",
        policy_challenges_by_seed[seed]
    )


    for task, records in policy_challenges_by_seed[seed].items():

        public_keys = {
            json.dumps(args, sort_keys=True)
            for args in public_inputs[task]
        }

        selected_keys = [
            json.dumps(record["input"], sort_keys=True)
            for record in records
        ]

        counts = {
            category: sum(
                record["category"] == category
                for record in records
            )
            for category in CHALLENGE_CATEGORIES
        }

        available_counts = [
            count
            for count in counts.values()
            if count > 0
        ]

        balanced = (
            len(available_counts) > 0
            and max(available_counts) - min(available_counts) <= 1
        )

        valid = (
            len(records) == CHALLENGE_TEST_COUNT
            and len(set(selected_keys)) == len(records)
            and not set(selected_keys).intersection(public_keys)
            and all(
                record["reference_validated"]
                for record in records
            )
            and balanced
        )

        assert valid, (
            seed,
            task,
            counts,
            "Challenge validation failed"
        )

        challenge_validation.append({
            "seed": seed,
            "task_id": task,
            "valid": valid,
            **counts
        })



save_csv(
    "results/challenge_balance.csv",
    challenge_validation
)

print("\n" + "=" * 65)
print("CHALLENGE TEST VALIDATION COMPLETED")
print("=" * 65)

display_table(
    challenge_validation,
    "Validated challenge suites"
)

print(
    "\nValidated challenge suites:",
    len(challenge_validation),
    "task-seed pairs"
)

print("\nChallenge counts per seed:")

for seed in SEEDS:

    total = sum(
        len(records)
        for records in policy_challenges_by_seed[seed].values()
    )

    print(
        f"Seed {seed}: "
        f"{total} policy-aware challenge tests"
    )

print("\nAll challenge suites passed validation!")

BAGR CHALLENGE TEST GENERATION

Generating challenge tests for seed=17

Generating challenge tests for seed=42

Generating challenge tests for seed=73

CHALLENGE TEST VALIDATION COMPLETED
Validated challenge suites


,seed,task_id,valid,boundary,local,scale,structure
0,17,HumanEval/0,True,2,2,2,2
1,17,HumanEval/1,True,2,2,2,2
2,17,HumanEval/2,True,2,2,2,2
3,17,HumanEval/3,True,2,2,2,2
4,17,HumanEval/4,True,2,2,2,2
...,...,...,...,...,...,...,...
67,73,HumanEval/19,True,2,2,2,2
68,73,HumanEval/20,True,2,2,2,2
69,73,HumanEval/21,True,0,3,3,2
70,73,HumanEval/22,True,2,2,2,2



Validated challenge suites: 72 task-seed pairs

Challenge counts per seed:
Seed 17: 192 policy-aware challenge tests
Seed 42: 192 policy-aware challenge tests
Seed 73: 192 policy-aware challenge tests

All challenge suites passed validation!


### 7 — Metamorphic stability and BAGR

Equal-output relationships are automatically discovered from distinct reference-validated inputs. If a task has fewer than two collisions, repeatability checks are used. No HumanEval+ answers enter the reward.


In [ ]:
def build_metamorphic_tests(task, challenges):

    candidate_inputs = copy.deepcopy(public_inputs[task])
    candidate_inputs.extend(
        copy.deepcopy(record['input']) for record in challenges[task]
    )
    tests, seen = [], set()
    for arguments in candidate_inputs:
        key = json.dumps(arguments, sort_keys=True, ensure_ascii=False)
        if key in seen:
            continue
        seen.add(key)
        reference = execute_candidate(
            task, problems[task]['canonical_solution'], [arguments]
        )
        if reference[0]['status'] != 'ok':
            continue
        tests.append({
            'relation': 'repeatability',
            'inputs': [copy.deepcopy(arguments), copy.deepcopy(arguments)]
        })
        if len(tests) == 2:
            break
    if len(tests) < 2:
        raise ValueError(f'{task}: fewer than two distinct reference-valid inputs.')
    return tests


def relation_passes(test, results):
    return (
        test.get('relation') == 'repeatability'
        and len(results) == 2
        and all(result['status'] == 'ok' for result in results)
        and results[0]['output'] == results[1]['output']
    )


def build_metamorphic_suite(challenges):
    suite = {task: build_metamorphic_tests(task, challenges) for task in TASK_IDS}
    for task, tests in suite.items():
        for test in tests:
            reference = execute_candidate(
                task, problems[task]['canonical_solution'], test['inputs']
            )
            assert relation_passes(test, reference), (task, test, reference)
    return suite

### 8 — E1–E6 experiment families and matched reward adapters

E6 is the component-ablation family (`E6_noPw`, `E6_noPc`, `E6_noMw`, `E6_noV`). Every E2–E6 variant uses the same model, LoRA shape, batch/generation settings, optimizer, stages, and scheduled steps.


In [17]:
METHODS={
 'E2':{'name':'Weak-only GRPO','adaptive':False,'weights':{'Pw':1.0}},
 'E3':{'name':'Weak + static challenges','adaptive':False,'weights':{'Pw':.4,'Pc':.6}},
 'E4':{'name':'Weak + policy-aware challenges','adaptive':True,'weights':{'Pw':.4,'Pc':.6}},
 'E5':{'name':'Full BAGR','adaptive':True,'weights':WEIGHTS},
}
for removed in WEIGHTS:
    remaining={name:weight for name,weight in WEIGHTS.items() if name!=removed}
    total=sum(remaining.values())
    METHODS['E6_no'+removed]={'name':'BAGR without '+removed,'adaptive':True,
                             'weights':{name:weight/total for name,weight in remaining.items()}}

reward_log=[]
def completion_text(completion):
    if isinstance(completion,list):
        return '\n'.join(item.get('content','') for item in completion if isinstance(item,dict))
    if isinstance(completion,dict):
        return str(completion.get('content',''))
    return str(completion)

def make_reward(method,seed,stage,challenges,metamorphic_suite):
    weights=METHODS[method]['weights']
    call_counter={'value':0}

    def reward(completions,task_id,trainer_state=None,**kwargs):
        tasks=[task_id]*len(completions) if isinstance(task_id,str) else list(task_id)
        assert len(tasks)==len(completions),'Task/completion alignment error'
        call_counter['value']+=1
        call_id=call_counter['value']
        step=int(trainer_state.global_step) if trainer_state is not None else 0

        values=[]
        components=[]
        for task,completion in zip(tasks,completions):
            text=completion_text(completion)
            parts=calculate_bagr(task,text,challenges,metamorphic_suite)
            value=float(sum(weight*parts[name] for name,weight in weights.items()))
            values.append(value)
            components.append((task,text,parts))

        group_std=float(statistics.pstdev(values)) if len(values)>1 else 0.0
        group_unique=len(set(round(value,10) for value in values))
        event_path=OUTPUT_DIR/'logs'/'reward_events.jsonl'
        event_path.parent.mkdir(parents=True,exist_ok=True)

        with event_path.open('a',encoding='utf-8') as handle:
            for index,((task,text,parts),value) in enumerate(zip(components,values)):
                row={
                    'method':method,'seed':seed,'stage':stage,'task_id':task,
                    'step':step,'reward_call':call_id,'completion_index':index,
                    'completion':text,'reward':value,
                    'group_reward_std':group_std,
                    'group_unique_rewards':group_unique,
                    **parts
                }
                reward_log.append(row)
                handle.write(json.dumps(row,ensure_ascii=False,allow_nan=False)+'\\n')

        return values

    reward.__name__=method+'_reward'
    return reward
display_table([{'Experiment':method,'Description':spec['name'],'Weights':str(spec['weights'])}
               for method,spec in METHODS.items()],'E2–E6 trained variants')


E2–E6 trained variants


,Experiment,Description,Weights
0,E2,Weak-only GRPO,{'Pw': 1.0}
1,E3,Weak + static challenges,"{'Pw': 0.4, 'Pc': 0.6}"
2,E4,Weak + policy-aware challenges,"{'Pw': 0.4, 'Pc': 0.6}"
3,E5,Full BAGR,"{'Pw': 0.4, 'Pc': 0.35, 'Mw': 0.15, 'V': 0.1}"
4,E6_noPw,BAGR without Pw,"{'Pc': 0.5833333333333334, 'Mw': 0.25, 'V': 0...."
5,E6_noPc,BAGR without Pc,"{'Pw': 0.6153846153846154, 'Mw': 0.23076923076..."
6,E6_noMw,BAGR without Mw,"{'Pw': 0.4705882352941177, 'Pc': 0.41176470588..."
7,E6_noV,BAGR without V,"{'Pw': 0.4444444444444445, 'Pc': 0.38888888888..."


### 9 — BF16/AMP-safe LoRA trainer and quantitative update audit

The frozen base model stays in BF16 or FP16, while every trainable LoRA parameter is converted to FP32 after `GRPOTrainer` initialization and before the optimizer is created. FP16 alone uses a gradient scaler; BF16 does not. Each stage records the LoRA L2 delta, maximum absolute delta, changed tensor count, changed scalar count, and skipped optimizer steps. Training fails loudly if a final adapter did not actually change.


In [18]:
class FP32LoRAGRPOTrainer(GRPOTrainer):
    def __init__(self,*args,**kwargs):
        super().__init__(*args,**kwargs)
        with torch.no_grad():
            for parameter in self.model.parameters():
                if parameter.requires_grad:
                    parameter.data=parameter.data.to(torch.float32)
        self.model.zero_grad(set_to_none=True)
        trainable=[parameter for parameter in self.model.parameters() if parameter.requires_grad]
        assert trainable and all(parameter.dtype==torch.float32 for parameter in trainable)

class OptimizerAudit(TrainerCallback):
    def __init__(self,accelerator):
        self.accelerator=accelerator
        self.applied=0
        self.skipped=0
    def on_optimizer_step(self,args,state,control,**kwargs):
        skipped=bool(getattr(self.accelerator,'optimizer_step_was_skipped',False))
        self.skipped+=int(skipped)
        self.applied+=int(not skipped)

def attach_lora(model):
    config=LoraConfig(r=8,lora_alpha=16,lora_dropout=0.0,bias='none',task_type='CAUSAL_LM',
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
    return get_peft_model(model,config)

def lora_parameters(model,trainable_only=False):
    return {name:parameter for name,parameter in model.named_parameters()
            if 'lora_' in name and (parameter.requires_grad or not trainable_only)}

def adapter_fingerprint(model):
    digest=hashlib.sha256()
    parameters=lora_parameters(model)
    assert parameters
    for name,parameter in sorted(parameters.items()):
        digest.update(name.encode())
        digest.update(parameter.detach().float().cpu().numpy().tobytes())
    return digest.hexdigest()

def trainable_snapshot(model):
    parameters=lora_parameters(model,trainable_only=True)
    assert parameters and all(parameter.dtype==torch.float32 for parameter in parameters.values())
    return {name:parameter.detach().float().cpu().clone() for name,parameter in parameters.items()}

def parameter_delta(before,model):
    after=lora_parameters(model,trainable_only=True)
    squared=0.0
    maximum=0.0
    changed_tensors=0
    changed_values=0
    for name,old in before.items():
        difference=after[name].detach().float().cpu()-old
        squared+=float(torch.sum(difference*difference))
        maximum=max(maximum,float(difference.abs().max()))
        changed=int(torch.count_nonzero(difference))
        changed_values+=changed
        changed_tensors+=int(changed>0)
    return {'LoRA delta L2':math.sqrt(squared),'LoRA delta max':maximum,
            'Changed LoRA tensors':changed_tensors,'Changed LoRA values':changed_values}

parameter=torch.nn.Parameter(torch.ones(4,dtype=torch.float32,device=DEVICE))
optimizer=torch.optim.SGD([parameter],lr=.01)
autocast_enabled=DEVICE=='cuda'
autocast_dtype=torch.bfloat16 if USE_BF16 else torch.float16
if USE_FP16:
    scaler=torch.amp.GradScaler('cuda')
    with torch.autocast(device_type='cuda',dtype=autocast_dtype,enabled=autocast_enabled):
        loss=parameter.square().mean()
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
elif USE_BF16:
    with torch.autocast(device_type='cuda',dtype=autocast_dtype):
        loss=parameter.square().mean()
    loss.backward()
    optimizer.step()
else:
    parameter.square().mean().backward()
    optimizer.step()
assert parameter.grad.dtype==torch.float32 and torch.isfinite(parameter.grad).all()
del parameter,optimizer
clear_device_cache()
print('Mixed-precision FP32-gradient smoke check: PASS; bf16=',USE_BF16,'fp16=',USE_FP16)


Mixed-precision FP32-gradient smoke check: PASS; bf16= True fp16= False


In [19]:
from pathlib import Path

print("Kaggle output directory:", OUTPUT_DIR.resolve())
print("This directory is under /kaggle/working and will be available in the notebook Output panel.")
print("After the run finishes, use Save Version with outputs enabled to preserve the results.")


Kaggle output directory: /kaggle/working/BAGR_project_results/20261102_125023_584374
This directory is under /kaggle/working and will be available in the notebook Output panel.
After the run finishes, use Save Version with outputs enabled to preserve the results.


In [ ]:


import ast
import textwrap
import statistics

BAGR_WEIGHTS = {
    "Pw": 0.40,
    "Pc": 0.35,
    "Mw": 0.15,
    "V": 0.10
}


def calculate_bagr(task, completion, challenges, metamorphic_suite):


    

    Pw = 0.0
    Pc = 0.0
    Mw = 0.0
    V = 0.0

    
    try:
        source = complete_source(task, completion)

        tree = ast.parse(textwrap.dedent(source))

        entry_point = problems[task]["entry_point"]

        function_found = any(
            isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
            and node.name == entry_point
            for node in tree.body
        )

        V = float(function_found)

    except (SyntaxError, ValueError, TypeError):
        V = 0.0

    
    if V == 0.0:
        return {
            "Pw": 0.0,
            "Pc": 0.0,
            "Mw": 0.0,
            "V": 0.0,
            "bagr_reward": 0.0
        }

    
    try:
        weak_score = weak_correctness(
            task,
            completion,
            WEAK_TEST_COUNT
        )

        Pw = (
            float(weak_score)
            if weak_score is not None
            else 0.0
        )

    except Exception:
        Pw = 0.0

    

    try:
        task_challenges = challenges.get(task, [])

        if task_challenges:

            challenge_inputs = [
                record["input"]
                for record in task_challenges
            ]

            expected_outputs = [
                record["expected_output"]
                for record in task_challenges
            ]

            Pc, _ = correctness(
                task,
                completion,
                challenge_inputs,
                expected_outputs
            )

            Pc = float(Pc)

    except Exception:
        Pc = 0.0


    try:
        task_tests = metamorphic_suite.get(task, [])

        passed = 0
        total = len(task_tests)

        for test in task_tests:

            test_inputs = test["inputs"]

            results = execute_candidate(
                task,
                completion,
                test_inputs
            )

            if relation_passes(test, results):
                passed += 1

        Mw = (
            float(passed / total)
            if total > 0
            else 0.0
        )

    except Exception:
        Mw = 0.0


    components = {
        "Pw": Pw,
        "Pc": Pc,
        "Mw": Mw,
        "V": V
    }

    bagr_reward = sum(
        BAGR_WEIGHTS[name] * components[name]
        for name in BAGR_WEIGHTS
    )
    bagr_reward = max(0.0, min(1.0, float(bagr_reward)))

    return {
        "Pw": float(Pw),
        "Pc": float(Pc),
        "Mw": float(Mw),
        "V": float(V),
        "bagr_reward": bagr_reward
    }


print("=" * 60)
print("BAGR REWARD FUNCTION INITIALIZED")
print("=" * 60)

print("Reward weights:", BAGR_WEIGHTS)
print("calculate_bagr available:", callable(calculate_bagr))

assert abs(sum(BAGR_WEIGHTS.values()) - 1.0) < 1e-9

print("Total reward weights:", sum(BAGR_WEIGHTS.values()))


BAGR REWARD FUNCTION INITIALIZED
Reward weights: {'Pw': 0.4, 'Pc': 0.35, 'Mw': 0.15, 'V': 0.1}
calculate_bagr available: True
Total reward weights: 1.0


In [ ]:


task = TASK_IDS[0]

reference_completion = problems[task]["canonical_solution"]

test_result = calculate_bagr(
    task,
    reference_completion,
    policy_challenges_by_seed[SEEDS[0]],
    build_metamorphic_suite(
        policy_challenges_by_seed[SEEDS[0]]
    )
)

print("Task:", task)
print("Reference reward components:")

for name in ["Pw", "Pc", "Mw", "V", "bagr_reward"]:
    print(f"{name}: {test_result[name]:.4f}")

assert test_result["Pw"] == 1.0
assert test_result["Pc"] == 1.0
assert test_result["Mw"] == 1.0
assert test_result["V"] == 1.0
assert abs(test_result["bagr_reward"] - 1.0) < 1e-9

print("\nReference solution BAGR validation PASSED!")


Task: HumanEval/0
Reference reward components:
Pw: 1.0000
Pc: 1.0000
Mw: 1.0000
V: 1.0000
bagr_reward: 1.0000

Reference solution BAGR validation PASSED!


### 10 — Train all E2–E6 variants for all seeds, save every optimizer checkpoint, and reload saved adapters

The configuration factory is shared by every method. Evaluation samples are generated only after the final adapter is saved, unloaded, reloaded into a fresh base model, and fingerprint-matched to the in-memory adapter.


In [ ]:

print("E1 is baseline only; GRPO training begins at E2.")
print("Training variants:", list(METHODS))
print("Completions per prompt:", NUM_GENERATIONS)
print("Weak tests:", WEAK_TEST_COUNT)
print("Challenge tests:", CHALLENGE_TEST_COUNT)

_preview = training_config(SEEDS[0], "E2", 1)
print("Actual save strategy:", _preview.save_strategy)
print("Actual save steps:", _preview.save_steps)
print("Actual max steps:", _preview.max_steps)
assert _preview.save_strategy == "steps" and _preview.save_steps == 1
print("Reward events will be appended to:", OUTPUT_DIR / "logs" / "reward_events.jsonl")

E1 is baseline only; GRPO training begins at E2.
Training variants: ['E2', 'E3', 'E4', 'E5', 'E6_noPw', 'E6_noPc', 'E6_noMw', 'E6_noV']
Completions per prompt: 4
Weak tests: 1
Challenge tests: 8
Actual save strategy: SaveStrategy.STEPS
Actual save steps: 1
Actual max steps: 5
Reward events will be appended to: /kaggle/working/BAGR_project_results/20261102_125023_584374/logs/reward_events.jsonl


In [ ]:


def training_config(seed, method, stage):
    return GRPOConfig(
        output_dir=str(
            OUTPUT_DIR / 'checkpoints' /
            f'seed_{seed}' / method / f'stage_{stage}'
        ),

        max_steps=STEPS_PER_STAGE,
        learning_rate=5e-6,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_generations=NUM_GENERATIONS,
        max_prompt_length=768,
        max_completion_length=MAX_NEW_TOKENS,

        temperature=.8,
        top_p=.95,
        top_k=0,
        repetition_penalty=1.0,

        fp16=USE_FP16,
        bf16=USE_BF16,
        use_cpu=DEVICE == 'cpu',
        dataloader_pin_memory=DEVICE == 'cuda',
        gradient_checkpointing=False,

        beta=.02,
        loss_type='grpo',
        scale_rewards='group',
        optim='adamw_torch',
        lr_scheduler_type='constant',
        max_grad_norm=1.0,

        logging_steps=1,
        logging_first_step=True,

       
        save_strategy='steps',
        save_steps=1,
        save_total_limit=None,
        save_only_model=True,

        report_to='none',
        remove_unused_columns=False,
        seed=seed + stage - 1,
        data_seed=seed,
        use_vllm=False
    )



budget_signature = {
    'steps_per_stage': STEPS_PER_STAGE,
    'stages': STAGES,
    'learning_rate': 5e-6,
    'batch_size': 1,
    'gradient_accumulation': 4,
    'num_generations': NUM_GENERATIONS,
    'max_prompt_length': 768,
    'max_completion_length': MAX_NEW_TOKENS,
    'optimizer': 'adamw_torch',
    'scheduler': 'constant',
    'beta': .02,
    'loss_type': 'grpo',
    'scale_rewards': 'group',
    'lora_r': 8,
    'lora_alpha': 16,
    'checkpoint_save_strategy': 'steps',
    'checkpoint_save_steps': 1,
    'checkpoint_save_total_limit': None,
    'checkpoint_save_only_model': True
}

save_json('training_budget.json', budget_signature)




all_candidates = []

for seed in SEEDS:
    all_candidates.extend(baseline_candidates_by_seed[seed])

training_rows = []
update_rows = []

initial_fingerprints = {seed: [] for seed in SEEDS}



for seed in SEEDS:

    for method, spec in METHODS.items():

        print(f'\nseed={seed} {method}: {spec["name"]}')

        set_seed(seed)

        model = attach_lora(load_base_model())

        selected = copy.deepcopy(
            policy_challenges_by_seed[seed]
            if spec['adaptive']
            else static_challenges_by_seed[seed]
        )

        metamorphic_suite = build_metamorphic_suite(selected)

        current_probes = baseline_probes_by_seed[seed]

        total_steps = 0
        applied_updates = 0
        amp_skips = 0

        reset_memory_stats()

        started = time.perf_counter()

        initial_fingerprint = None


        for stage in range(1, STAGES + 1):

            if stage > 1 and spec['adaptive']:

                pool = {
                    task: generate_challenge_pool(
                        task,
                        seed * 100000 + stage * 1000 + index
                    )
                    for index, task in enumerate(TASK_IDS)
                }

                selected = choose_challenges(pool, current_probes)

                metamorphic_suite = build_metamorphic_suite(selected)

            save_json(
                f'data/seed_{seed}/{method}_stage_{stage}_challenges.json',
                selected
            )


            trainer = FP32LoRAGRPOTrainer(
                model=model,
                args=training_config(seed, method, stage),
                reward_funcs=make_reward(
                    method,
                    seed,
                    stage,
                    selected,
                    metamorphic_suite
                ),
                train_dataset=train_dataset,
                processing_class=tokenizer
            )


         
            audit = OptimizerAudit(trainer.accelerator)

            trainer.add_callback(audit)

            if initial_fingerprint is None:

                initial_fingerprint = adapter_fingerprint(
                    trainer.model
                )

                initial_fingerprints[seed].append(
                    initial_fingerprint
                )


            before = trainable_snapshot(trainer.model)


            print(
                f'\nStarting seed={seed}, '
                f'method={method}, stage={stage}'
            )

            print(
                f'Every optimizer step will be checkpointed under: '
                f'{OUTPUT_DIR / "checkpoints" / f"seed_{seed}" / method / f"stage_{stage}"}'
            )

            trainer.train()


            stage_output = (
                OUTPUT_DIR / 'checkpoints' /
                f'seed_{seed}' / method / f'stage_{stage}'
            )
            step_checkpoints = sorted(
                stage_output.glob('checkpoint-*'),
                key=lambda path: int(path.name.split('-')[-1])
            )
            checkpoint_steps = [
                int(path.name.split('-')[-1])
                for path in step_checkpoints
            ]
            if checkpoint_steps != list(range(1, STEPS_PER_STAGE + 1)):
                raise RuntimeError(
                    f'Missing step checkpoints for seed={seed}, '
                    f'method={method}, stage={stage}: {checkpoint_steps}'
                )

            save_json(
                f'checkpoints/seed_{seed}/{method}/stage_{stage}_status.json',
                {
                    'seed': seed,
                    'method': method,
                    'stage': stage,
                    'global_step': int(trainer.state.global_step),
                    'checkpoint_steps': checkpoint_steps,
                    'checkpoint_directory': str(stage_output.resolve()),
                    'completed': True
                }
            )
            save_checkpoint_manifest()


            assert trainer.state.global_step == STEPS_PER_STAGE

            assert (
                audit.applied + audit.skipped
                == STEPS_PER_STAGE
            )



            delta = parameter_delta(
                before,
                trainer.model
            )

            update_rows.append({
                'seed': seed,
                'Experiment': method,
                'Stage': stage,
                'Applied updates': audit.applied,
                'AMP-skipped updates': audit.skipped,
                **delta
            })


            total_steps += int(trainer.state.global_step)

            applied_updates += audit.applied

            amp_skips += audit.skipped


            history = [
                {
                    key: (
                        str(value)
                        if isinstance(value, float)
                        and not math.isfinite(value)
                        else value
                    )
                    for key, value in row.items()
                }
                for row in trainer.state.log_history
            ]

            save_json(
                f'logs/seed_{seed}/{method}_stage_{stage}.json',
                history
            )


            trainer.save_model()

            model = trainer.model


            if stage < STAGES:

                current_probes = sample_policy(
                    model,
                    PROBE_SAMPLES_PER_TASK,
                    seed + 2000 + stage * 100,
                    method + '_probe'
                )

                save_candidates(
                    f'candidates/seed_{seed}/{method}_stage_{stage}_probes.jsonl',
                    current_probes
                )



            del audit, trainer, before

            clear_device_cache()


        final_fingerprint = adapter_fingerprint(model)

        changed = final_fingerprint != initial_fingerprint

        if (
            not changed
            or applied_updates == 0
            or sum(
                row['Changed LoRA values']
                for row in update_rows
                if row['seed'] == seed
                and row['Experiment'] == method
            ) == 0
        ):

            raise RuntimeError(
                f'{method} seed {seed}: '
                f'no verified LoRA parameter update'
            )



        final_path = (
            OUTPUT_DIR /
            'checkpoints' /
            f'seed_{seed}' /
            method /
            'final_adapter'
        )

        model.save_pretrained(final_path)

        tokenizer.save_pretrained(final_path)

        save_json(
            f'checkpoints/seed_{seed}/{method}/final_adapter_status.json',
            {
                'seed': seed,
                'method': method,
                'adapter_path': str(final_path.resolve()),
                'fingerprint': final_fingerprint,
                'completed': True
            }
        )
        save_checkpoint_manifest()


        del model

        clear_device_cache()


        reloaded = PeftModel.from_pretrained(
            load_base_model(),
            final_path,
            is_trainable=False
        ).to(DEVICE)

        reloaded_fingerprint = adapter_fingerprint(reloaded)

        if reloaded_fingerprint != final_fingerprint:

            raise RuntimeError(
                f'{method} seed {seed}: '
                f'saved adapter fingerprint mismatch'
            )


        samples = sample_policy(
            reloaded,
            EVAL_SAMPLES_PER_TASK,
            seed + 1000,
            method
        )

        all_candidates.extend(samples)

        save_candidates(
            f'candidates/seed_{seed}/{method}.jsonl',
            samples
        )

        elapsed = time.perf_counter() - started

        training_rows.append({
            'seed': seed,
            'Experiment': method,
            'Steps': total_steps,
            'Applied updates': applied_updates,
            'AMP-skipped updates': amp_skips,
            'Seconds': elapsed,
            'Device': DEVICE,
            'Base dtype': str(MODEL_DTYPE),
            'Adapter changed': changed,
            'Saved adapter verified': True,
            **memory_stats()
        })


        save_csv(
            'results/training_efficiency.csv',
            training_rows
        )

        save_csv(
            'results/lora_update_audit.csv',
            update_rows
        )

        save_csv(
            'logs/reward_components.csv',
            reward_log
        )


        del reloaded

        clear_device_cache()


for seed, fingerprints in initial_fingerprints.items():

    assert len(set(fingerprints)) == 1, (
        f'Seed {seed} methods did not share '
        f'the same LoRA initialization'
    )


assert all(
    row['Steps'] == STAGES * STEPS_PER_STAGE
    for row in training_rows
)

assert all(
    row['Adapter changed']
    and row['Saved adapter verified']
    for row in training_rows
)


display_table(
    training_rows,
    'Completed matched-budget training with verified saved LoRA updates'
)

save_checkpoint_manifest()
print("\nAll GRPO experiments completed successfully.")
print("Every optimizer-step checkpoint, stage adapter, and final adapter is saved.")
print("Checkpoint manifest:", (OUTPUT_DIR / 'checkpoint_manifest.json').resolve())



seed=17 E2: Weak-only GRPO


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.



Starting seed=17, method=E2, stage=1
Every optimizer step will be checkpointed under: /kaggle/working/BAGR_project_results/20261102_125023_584374/checkpoints/seed_17/E2/stage_1


Step,Training Loss
1,0.000000
2,0.000000
3,0.000000
4,0.000000
5,0.000000



Starting seed=17, method=E2, stage=2
Every optimizer step will be checkpointed under: /kaggle/working/BAGR_project_results/20261102_125023_584374/checkpoints/seed_17/E2/stage_2


Step,Training Loss
1,0.000000
2,0.000000
3,-0.000000
4,0.000000
5,0.000000


... (56 outputs left out)


Starting seed=42, method=E4, stage=1
Every optimizer step will be checkpointed under: /kaggle/working/BAGR_project_results/20261102_125023_584374/checkpoints/seed_42/E4/stage_1


Step,Training Loss
1,-0.000000
2,-0.000000
3,0.000000
4,0.000000
5,0.000000



Starting seed=42, method=E4, stage=2
Every optimizer step will be checkpointed under: /kaggle/working/BAGR_project_results/20261102_125023_584374/checkpoints/seed_42/E4/stage_2


Step,Training Loss
1,0.000000
2,0.000000
3,0.000000
4,0.000000
5,-0.000000


In [ ]:


from pathlib import Path
import json

candidate_root = OUTPUT_DIR / "candidates"
recovered_candidates = []

for path in sorted(candidate_root.glob("seed_*/*.jsonl")):
   
    if "probe" in path.name:
        continue
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                record = json.loads(line)
                if "completion" not in record:
                    raise ValueError(f"Missing completion in {path}")
                recovered_candidates.append(record)

expected_cohorts = {
    (method, seed)
    for method in ["E1", *METHODS.keys()]
    for seed in SEEDS
}
found_cohorts = {
    (record["method"], int(record["seed"]))
    for record in recovered_candidates
}
missing_cohorts = sorted(expected_cohorts - found_cohorts)

if missing_cohorts:
    raise RuntimeError(
        "The saved candidate files are incomplete. Missing cohorts: "
        + repr(missing_cohorts)
        + ". Finish the missing training runs before final evaluation."
    )

for method, seed in sorted(expected_cohorts):
    rows = [
        row for row in recovered_candidates
        if row["method"] == method and int(row["seed"]) == seed
    ]
    expected_pairs = {
        (task, sample_id)
        for task in TASK_IDS
        for sample_id in range(EVAL_SAMPLES_PER_TASK)
    }
    actual_pairs = {
        (row["task_id"], int(row["sample_id"]))
        for row in rows
    }
    if actual_pairs != expected_pairs:
        raise RuntimeError(
            f"Incomplete or duplicate evaluation grid for {method}, seed={seed}"
        )

all_candidates = recovered_candidates

missing_adapters = []
for method in METHODS:
    for seed in SEEDS:
        adapter_dir = (
            OUTPUT_DIR / "checkpoints" /
            f"seed_{seed}" / method / "final_adapter"
        )
        if not (adapter_dir / "adapter_config.json").exists():
            missing_adapters.append(str(adapter_dir))

if missing_adapters:
    raise RuntimeError(
        "Missing final adapters required for provenance checks: "
        + repr(missing_adapters)
    )

manifest_records = save_checkpoint_manifest()

print("Recovered evaluation candidates:", len(all_candidates))
print("Complete method/seed cohorts:", len(found_cohorts))
print("Expected candidates:", len(expected_cohorts) * len(TASK_IDS) * EVAL_SAMPLES_PER_TASK)
print("Saved checkpoint files indexed:", len(manifest_records))
print("All final adapters found.")
print("Ready to run the independent official HumanEval+ evaluation cells.")


### 11 — Independent official EvalPlus evaluation

HumanEval+ is loaded only inside this isolated worker, after all checkpoints are frozen. Candidate cohorts are checked independently for every method and seed before official `evalplus.evaluate.check_correctness` calls are submitted.


In [ ]:
EVALPLUS_WORKER_CODE = r"""
import hashlib,json,sys,os,time
from contextlib import contextmanager
import evalplus.eval as eval_module
from pathlib import Path
from collections import Counter
from concurrent.futures import ProcessPoolExecutor
from evalplus.data import get_human_eval_plus,get_human_eval_plus_hash
from evalplus.evaluate import get_groundtruth,check_correctness

ORIGINAL_UNSAFE_EXECUTE=eval_module.unsafe_execute
ORIGINAL_GUARD=eval_module.reliability_guard

@contextmanager
def windows_time_limit(seconds):
    started=time.perf_counter()
    try:
        yield
    finally:
        if time.perf_counter()-started>seconds:
            raise TimeoutError('Input exceeded EvalPlus time budget')

def windows_guard(maximum_memory_bytes=None):
    ORIGINAL_GUARD(maximum_memory_bytes=None)

def windows_unsafe_execute(*args):
    eval_module.time_limit=windows_time_limit
    eval_module.reliability_guard=windows_guard
    return ORIGINAL_UNSAFE_EXECUTE(*args)

def platform_check_correctness(*args):
    if os.name=='nt':
        eval_module.unsafe_execute=windows_unsafe_execute
    return check_correctness(*args)

def main():
    request=json.loads(Path(sys.argv[1]).read_text(encoding='utf-8'))
    full=get_human_eval_plus()
    tasks=request['tasks']
    selected={task:full[task] for task in tasks}
    dataset_hash=str(get_human_eval_plus_hash())
    subset_hash=hashlib.sha256(json.dumps(sorted(tasks)).encode()).hexdigest()[:12]
    expected=get_groundtruth(selected,dataset_hash+'_bagr_'+subset_hash,[])
    grouped={}
    for candidate in request['candidates']:
        grouped.setdefault((candidate['method'],candidate['seed']),[]).append(candidate)
    for cohort,records in grouped.items():
        counts=Counter(record['task_id'] for record in records)
        if set(counts)!=set(tasks) or set(counts.values())!={request['samples_per_task']}:
            raise ValueError('Unequal evaluation cohort: '+repr(cohort))
        for task in tasks:
            sample_ids=[record['sample_id'] for record in records if record['task_id']==task]
            if sorted(sample_ids)!=list(range(request['samples_per_task'])):
                raise ValueError('Missing or duplicate sample identifiers: '+repr((cohort,task)))
    futures=[]
    rows=[]
    workers=max(1,min(4,os.cpu_count() or 1))
    with ProcessPoolExecutor(max_workers=workers) as pool:
        for candidate in request['candidates']:
            task=candidate['task_id']
            identifier=f"{candidate['method']}:{candidate['seed']}:{task}:{candidate['sample_id']}"
            future=pool.submit(platform_check_correctness,'humaneval',candidate['sample_id'],
                selected[task],candidate['solution'],expected[task],False,False,identifier,1.0,4.0)
            futures.append((candidate,future))
        for candidate,future in futures:
            result=future.result()
            base,plus=result['base'][0],result['plus'][0]
            rows.append({'method':candidate['method'],'seed':candidate['seed'],
                'task_id':candidate['task_id'],'sample_id':candidate['sample_id'],
                'base_status':base,'plus_status':plus,'strong_pass':base==plus=='pass'})
    Path(request['output']).write_text(json.dumps({'dataset_hash':dataset_hash,'tasks':tasks,
        'checker':'evalplus.evaluate.check_correctness',
        'platform_adapter':'Windows elapsed-time checks + official watchdog; no setrlimit' if os.name=='nt' else 'none',
        'records':rows},indent=2),encoding='utf-8')
    print('Official HumanEval+ evaluation completed:',len(rows),'programs')

if __name__=='__main__':
    main()
"""

expected_methods={'E1',*METHODS}
assert {candidate['method'] for candidate in all_candidates}==expected_methods
manifest={'model_revision':MODEL_REVISION,'candidate_hashes':{},'adapter_hashes':{}}
for path in sorted((OUTPUT_DIR/'candidates').glob('seed_*/*.jsonl')):
    if 'probe' not in path.name:
        manifest['candidate_hashes'][str(path.relative_to(OUTPUT_DIR))]=hashlib.sha256(path.read_bytes()).hexdigest()
for path in sorted((OUTPUT_DIR/'checkpoints').glob('seed_*/*/final_adapter/adapter_model.safetensors')):
    manifest['adapter_hashes'][str(path.relative_to(OUTPUT_DIR))]=hashlib.sha256(path.read_bytes()).hexdigest()
save_json('pre_evaluation_manifest.json',manifest)
request={'tasks':TASK_IDS,'samples_per_task':EVAL_SAMPLES_PER_TASK,
    'candidates':[dict(candidate,solution=complete_source(candidate['task_id'],candidate['completion']))
                  for candidate in all_candidates],
    'output':str((OUTPUT_DIR/'results'/'official_humaneval_plus.json').resolve())}
with tempfile.TemporaryDirectory() as directory:
    worker=Path(directory)/'evalplus_worker.py'
    worker.write_text(EVALPLUS_WORKER_CODE,encoding='utf-8')
    request_file=Path(directory)/'request.json'
    request_file.write_text(json.dumps(request),encoding='utf-8')
    process=subprocess.run([sys.executable,'-X','utf8','-I',str(worker),str(request_file)],
        capture_output=True,text=True,encoding='utf-8',timeout=21600,env=worker_environment())
(OUTPUT_DIR/'logs'/'evalplus_stdout.log').write_text(process.stdout,encoding='utf-8')
(OUTPUT_DIR/'logs'/'evalplus_stderr.log').write_text(process.stderr,encoding='utf-8')
if process.returncode!=0:
    raise RuntimeError('Required HumanEval+ evaluation failed: '+process.stderr[-3000:])
official=json.loads((OUTPUT_DIR/'results'/'official_humaneval_plus.json').read_text(encoding='utf-8'))
print(process.stdout[-1500:])
print('Official dataset hash:',official['dataset_hash'])


### 12 — Per-seed metrics, mean/standard deviation, and method differences

All headline statistics are first computed independently within a seed. The final tables report the mean and sample standard deviation across seeds, plus paired seed-level differences for every pair of methods.


In [ ]:
statuses={(row['method'],row['seed'],row['task_id'],row['sample_id']):row
          for row in official['records']}
assert len(statuses)==len(all_candidates)
evaluation_rows=[]
coverage_rows=[]
for candidate in all_candidates:
    method,seed,task,index=(candidate['method'],candidate['seed'],candidate['task_id'],candidate['sample_id'])
    parts=calculate_bagr(task,candidate['completion'],reporting_challenges[seed],reporting_metamorphic[seed])
    status=statuses[(method,seed,task,index)]
    weak_pass=parts['Pw']==1
    evaluation_rows.append({'method':method,'seed':seed,'task_id':task,'sample_id':index,
        'weak_pass':weak_pass,'strong_pass':status['strong_pass'],
        'weak_pass_strong_fail':weak_pass and not status['strong_pass'],
        'base_status':status['base_status'],'plus_status':status['plus_status'],**parts,
        'token_limit_reached':candidate['token_limit_reached']})
    for count in [1,3,5,'all']:
        score=weak_correctness(task,candidate['completion'],count)
        coverage_rows.append({'method':method,'seed':seed,'task_id':task,'sample_id':index,
            'weak_tests':str(count),'eligible':score is not None,
            'weak_pass':score==1 if score is not None else None,'strong_pass':status['strong_pass']})

def metrics(rows):
    total=len(rows)
    weak=sum(row['weak_pass'] for row in rows)
    strong=sum(row['strong_pass'] for row in rows)
    hacks=sum(row['weak_pass'] and not row['strong_pass'] for row in rows)
    return {'Programs':total,'Weak pass (%)':100*weak/total,'Strong pass (%)':100*strong/total,
        'Gap (pp)':100*(weak-strong)/total,'Weak-pass count':weak,
        'Weak-pass/strong-fail':hacks,'Hack rate (%)':100*hacks/weak if weak else None}

method_order=['E1',*METHODS]
seed_metrics=[]
for method in method_order:
    for seed in SEEDS:
        rows=[row for row in evaluation_rows if row['method']==method and row['seed']==seed]
        seed_metrics.append({'Experiment':method,'seed':seed,**metrics(rows),
            'Mean challenge Pc':statistics.fmean(row['Pc'] for row in rows),
            'Mean metamorphic Mw':statistics.fmean(row['Mw'] for row in rows)})

def mean_std(values):
    values=[value for value in values if value is not None]
    return (statistics.fmean(values),statistics.stdev(values) if len(values)>1 else 0.0) if values else (None,None)

summary=[]
metric_names=['Weak pass (%)','Strong pass (%)','Gap (pp)','Hack rate (%)','Mean challenge Pc','Mean metamorphic Mw']
for method in method_order:
    rows=[row for row in seed_metrics if row['Experiment']==method]
    record={'Experiment':method,'Seeds':len(rows),'Programs per seed':rows[0]['Programs']}
    for name in metric_names:
        mean,standard_deviation=mean_std([row[name] for row in rows])
        record[name+' mean']=mean
        record[name+' std']=standard_deviation
    summary.append(record)

pairwise=[]
for left,right in itertools.combinations(method_order,2):
    for name in ['Weak pass (%)','Strong pass (%)','Gap (pp)','Hack rate (%)']:
        differences=[]
        for seed in SEEDS:
            left_row=next(row for row in seed_metrics if row['Experiment']==left and row['seed']==seed)
            right_row=next(row for row in seed_metrics if row['Experiment']==right and row['seed']==seed)
            if left_row[name] is not None and right_row[name] is not None:
                differences.append(right_row[name]-left_row[name])
        mean,standard_deviation=mean_std(differences)
        pairwise.append({'Left':left,'Right':right,'Metric':name,'Seeds':len(differences),
                         'Right - Left mean':mean,'Right - Left std':standard_deviation})

display_table(seed_metrics,'Independent per-seed official results')
display_table(summary,'Across-seed mean and standard deviation')
display_table([row for row in pairwise if row['Metric']=='Strong pass (%)'],
              'Paired strong-pass differences between methods')
save_csv('results/per_seed_metrics.csv',seed_metrics)
save_csv('results/multi_seed_summary.csv',summary)
save_csv('results/pairwise_method_differences.csv',pairwise)
save_csv('results/per_program_results.csv',evaluation_rows)
save_csv('results/weak_coverage_per_program.csv',coverage_rows)


### 13 — Charts and final report


In [ ]:
summary_frame=pd.DataFrame(summary).set_index('Experiment')
figure,axes=plt.subplots(1,3,figsize=(16,4.5))
summary_frame['Strong pass (%) mean'].plot.bar(yerr=summary_frame['Strong pass (%) std'],
    ax=axes[0],color='#278477',capsize=3)
axes[0].set_ylim(0,105);axes[0].set_title('Official HumanEval+ strong pass')
summary_frame['Gap (pp) mean'].plot.bar(yerr=summary_frame['Gap (pp) std'],
    ax=axes[1],color='#cf985b',capsize=3)
axes[1].axhline(0,color='black',linewidth=.7);axes[1].set_title('Weak–strong gap')
summary_frame['Hack rate (%) mean'].plot.bar(yerr=summary_frame['Hack rate (%) std'],
    ax=axes[2],color='#ae6464',capsize=3)
axes[2].set_ylim(0,105);axes[2].set_title('Conditional hack rate')
figure.suptitle(f'Qwen GRPO/LoRA · {len(TASK_IDS)} tasks · {len(SEEDS)} seeds · mean ± SD')
figure.tight_layout();figure.savefig(OUTPUT_DIR/'figures'/'multi_seed_comparison.png',dpi=150);plt.show()

lookup={row['Experiment']:row for row in summary}
e2,e5=lookup['E2'],lookup['E5']
reasons=[
    f'E5 strong pass is {e5["Strong pass (%) mean"]:.2f} ± {e5["Strong pass (%) std"]:.2f}% across seeds.',
    f'E2 strong pass is {e2["Strong pass (%) mean"]:.2f} ± {e2["Strong pass (%) std"]:.2f}% across seeds.',
    f'E5 - E2 strong-pass difference is {e5["Strong pass (%) mean"]-e2["Strong pass (%) mean"]:+.2f} percentage points.',
    'All E2–E6 variants used the same declared training budget; E1 is the untrained baseline.',
    'Every reported trained policy was sampled after reloading and fingerprint-verifying its saved LoRA adapter.',
    'Challenge suites were reference-validated and category-balanced for every selected task and seed.',
    'Official HumanEval+ was loaded only in the isolated post-training evaluation worker.'
]
for reason in reasons:
    print('•',reason)
save_json('results/reasoning.json',reasons)

report='# BAGR expanded GRPO/LoRA pilot\n\n'
report+=f'Model: {MODEL_NAME}; revision: {MODEL_REVISION}.\n\n'
report+=f'Tasks: {len(TASK_IDS)} ({TASK_IDS[0]} through {TASK_IDS[-1]}). Seeds: {SEEDS}.\n\n'
report+=f'Trained budget: {STAGES*STEPS_PER_STAGE} optimizer steps per E2–E6 variant and seed.\n\n'
report+=f'Precision: base={MODEL_DTYPE}, BF16={USE_BF16}, FP16={USE_FP16}; trainable LoRA parameters=FP32.\n\n'
report+=f'Official EvalPlus dataset hash: {official["dataset_hash"]}.\n\n'
report+='## Findings\n\n'+'\n\n'.join(reasons)+'\n\n'
report+='See multi_seed_summary.csv, pairwise_method_differences.csv, lora_update_audit.csv, challenge_balance.csv, and multi_seed_comparison.png.\n'
(OUTPUT_DIR/'results'/'REPORT.md').write_text(report,encoding='utf-8')
print('Completed expanded multi-seed pipeline.')
print('Report:',(OUTPUT_DIR/'results'/'REPORT.md').resolve())
print('Adapters:',(OUTPUT_DIR/'checkpoints').resolve())


In [ ]:

reward_events = OUTPUT_DIR / "logs" / "reward_events.jsonl"
if reward_events.exists():
    events = pd.read_json(reward_events, lines=True)
    print("Logged reward rows:", len(events))
    display(
        events.groupby(["method","seed","stage","task_id"]).agg(
            reward_count=("reward","size"),
            reward_mean=("reward","mean"),
            reward_std=("reward","std"),
            unique_rewards=("reward","nunique"),
            mean_group_std=("group_reward_std","mean"),
            min_reward=("reward","min"),
            max_reward=("reward","max")
        ).reset_index().tail(40)
    )
else:
    print("No reward events logged yet.")